Th goal of this assignment is to predict passenger survival based on characteristics such as age, sex, passenger class, fare, and embarkation port.

In [1]:
#Improt the necessary library and load the dataset
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

# Load Titanic dataset
df = sns.load_dataset("titanic")

# Display first five rows
print(df.head())

# Dataset dimensions
print("Dataset Shape:", df.shape)

   survived  pclass     sex   age  sibsp  parch     fare embarked  class  \
0         0       3    male  22.0      1      0   7.2500        S  Third   
1         1       1  female  38.0      1      0  71.2833        C  First   
2         1       3  female  26.0      0      0   7.9250        S  Third   
3         1       1  female  35.0      1      0  53.1000        S  First   
4         0       3    male  35.0      0      0   8.0500        S  Third   

     who  adult_male deck  embark_town alive  alone  
0    man        True  NaN  Southampton    no  False  
1  woman       False    C    Cherbourg   yes  False  
2  woman       False  NaN  Southampton   yes   True  
3  woman       False    C  Southampton   yes  False  
4    man        True  NaN  Southampton    no   True  
Dataset Shape: (891, 15)


**Select features and target variable**

In [4]:
# Select features
features = [
    "pclass", "sex", "age", "sibsp",
    "parch", "fare", "embarked"
]

X = df[features]
y = df["survived"]

**Explore the dataset**


In [3]:
# Dataset information
df.info()

# Numerical summary statistics
print(df.describe())


<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    str     
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    str     
 8   class        891 non-null    category
 9   who          891 non-null    str     
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    str     
 13  alive        891 non-null    str     
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), str(5)
memory usage: 100.4 KB
         survived      pclass         age       sibsp       parch        fare
count  891.0000

**Split the dataset BEFORE preprocessing**

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

Training set: (712, 7)
Testing set: (179, 7)


In [6]:
# Check missing values
print(df.isnull().sum())

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64


**Handle missing values using training dataset**

In [7]:
# Create copies
X_train = X_train.copy()
X_test = X_test.copy()

# Calculate median age from training data only
median_age = X_train["age"].median()

# Fill missing age values
X_train["age"] = X_train["age"].fillna(median_age)
X_test["age"] = X_test["age"].fillna(median_age)

# Calculate most frequent embarkation port from training data
mode_embarked = X_train["embarked"].mode()[0]

# Fill missing embarked values
X_train["embarked"] = X_train["embarked"].fillna(mode_embarked)
X_test["embarked"] = X_test["embarked"].fillna(mode_embarked)

# Check remaining missing values
print("Training missing values:")
print(X_train.isnull().sum())

print("\nTesting missing values:")
print(X_test.isnull().sum())

Training missing values:
pclass      0
sex         0
age         0
sibsp       0
parch       0
fare        0
embarked    0
dtype: int64

Testing missing values:
pclass      0
sex         0
age         0
sibsp       0
parch       0
fare        0
embarked    0
dtype: int64


**Encode Categorical Variables**

In [8]:
# One-hot encoding
X_train = pd.get_dummies(
    X_train,
    columns=["sex", "embarked"],
    drop_first=True,
    dtype=int
)

X_test = pd.get_dummies(
    X_test,
    columns=["sex", "embarked"],
    drop_first=True,
    dtype=int
)

# Ensure testing columns match training columns
X_test = X_test.reindex(
    columns=X_train.columns,
    fill_value=0
)

print(X_train.head())

     pclass   age  sibsp  parch      fare  sex_male  embarked_Q  embarked_S
692       3  28.5      0      0   56.4958         1           0           1
481       2  28.5      0      0    0.0000         1           0           1
527       1  28.5      0      0  221.7792         1           0           1
855       3  18.0      0      1    9.3500         0           0           1
801       2  31.0      1      1   26.2500         0           0           1


**Build a Logistic Regression Model**

In [9]:
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

**Model Evaluation**

In [11]:
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.8044692737430168

Confusion Matrix:
[[98 12]
 [23 46]]

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.89      0.85       110
           1       0.79      0.67      0.72        69

    accuracy                           0.80       179
   macro avg       0.80      0.78      0.79       179
weighted avg       0.80      0.80      0.80       179



The Logistic Regression model achieved an overall accuracy of 80%, meaning it correctly predicted the survival status of approximately 80% of the 179 passengers in the test dataset.

For class 0, (Those who did not survive)
Precision which means Of all passengers predicted not to survive, 81% actually did not survive
Recall meaning the model correctly identified 89% of passengers who did not survive.
F1-score meaning the model indicates strong performance in identifying passengers who did not survive at 85%


For Class 1 (Those who survide)
Precision meaning of all passengers predicted to survive, 79% actually survived.
Recall meaning the model correctly identified only 67% of actual survivors, meaning approximately 33% of survivors were incorrectly classified as not surviving. and F1-score of 72% indicates moderate performance in identifying survivors.

The logistic regression model was better at predicting passengers who did not survive (F1-score = 85%) than those who survived (F1-score = 72%). Note that this difference may be partly consistent with the class imbalance, as the test dataset contains more non-survivors (110) than survivors (69)